# CV面试题 10: NumPy图像处理与数学基础

本节涵盖计算机视觉面试中常见的 NumPy 操作和数学基础，包括：
- NumPy 数组操作 (shape, stride, 广播)
- 矩阵运算 (dot, matmul, einsum)
- 线性代数 (范数, SVD, 特征值分解)
- 傅里叶变换 (FFT, 频域滤波)
- 与图像处理相关的数学基础 (卷积, 互相关, 填充)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check_answer(question_num, answer, correct_answer, explanation=""):
    is_correct = answer == correct_answer
    status = "正确!" if is_correct else f"错误! 正确答案是 {correct_answer}"
    print(f"第{question_num}题: {status}")
    if explanation:
        print(f"  解析: {explanation}")
    return is_correct

## 一、选择题部分

---

### 第1题: np.ndarray 的 shape 和 stride 的关系

关于 NumPy ndarray 的 shape 和 stride (步幅)，以下说法**错误**的是？

A. shape 表示每个维度的大小，stride 表示在每个维度上移动到下一个元素需要跳过的字节数

B. 一个 shape 为 (3, 4) 的 float64 数组 (8字节)，其 strides 通常为 (32, 8)

C. 转置操作会创建新的内存副本并重新计算数据

D. 通过修改 stride 可以实现零拷贝的数组视图 (如 np.as_strided)

In [ ]:
# 第1题: 设置你的答案
answer = 'A'  # 修改这里

check_answer(1, answer, 'C', 
    '转置操作 (.T) 不会创建新的内存副本，'
    '它只是交换了 strides 和 shape，返回的是原数组的一个视图(view)。'
)

a = np.arange(12).reshape(3, 4)
print(f'原数组 shape: {a.shape}, strides: {a.strides}')
b = a.T
print(f'转置后 shape: {b.shape}, strides: {b.strides}')
print(f'是否共享内存: {np.shares_memory(a, b)}')

### 第2题: NumPy Broadcasting (广播) 的规则

以下哪组形状**不能**通过广播规则进行运算？

A. (3, 1) 和 (1, 4)

B. (5, 3, 4) 和 (3, 1)

C. (2, 3) 和 (3, 2)

D. (8, 1, 6, 1) 和 (7, 1, 5)

In [ ]:
# 第2题
answer = 'A'  # 修改这里

check_answer(2, answer, 'C', 
    '广播规则: 从后往前逐维比较，每维必须相同或其中一方为1。'
    '(2,3)和(3,2): 最后一维3和2不同且都不为1，无法广播。'
)

try:
    result = np.zeros((2, 3)) + np.zeros((3, 2))
    print('(2,3) + (3,2): 可以广播')
except ValueError as e:
    print(f'(2,3) + (3,2): 无法广播 - {e}')

### 第3题: np.dot vs np.matmul vs @ vs * 的区别

对于形状 (2, 3) 和 (3, 4) 的两个二维数组 A 和 B，以下哪个操作的结果形状与其他**不同**？

A. np.dot(A, B)

B. np.matmul(A, B)

C. A @ B

D. A * B

In [ ]:
# 第3题
answer = 'A'  # 修改这里

check_answer(3, answer, 'D', 
    'np.dot(A,B), np.matmul(A,B), A @ B 都是矩阵乘法，结果形状 (2, 4)。'
    'A * B 是逐元素乘法，需要形状相同或可广播，(2,3)*(3,4) 无法运算。'
)

A = np.random.randn(2, 3)
B = np.random.randn(3, 4)
print(f'np.dot(A,B) shape: {np.dot(A, B).shape}')
print(f'np.matmul(A,B) shape: {np.matmul(A, B).shape}')
print(f'A @ B shape: {(A @ B).shape}')
try:
    print(f'A * B shape: {(A * B).shape}')
except ValueError as e:
    print(f'A * B: 报错 - {e}')

### 第4题: np.einsum 的作用

以下哪个 np.einsum 表达式等价于矩阵乘法 C = A @ B (其中 A: (m,k), B: (k,n))？

A. np.einsum('ik,kj->ij', A, B)

B. np.einsum('ij,jk->ik', A, B)

C. np.einsum('ki,kj->ij', A, B)

D. np.einsum('ik,jk->ij', A, B)

In [ ]:
# 第4题
answer = 'A'  # 修改这里

check_answer(4, answer, 'A', 
    "np.einsum 使用爱因斯坦求和约定。'ik,kj->ij' 表示: "
    '对下标 k 进行求和，C[i,j] = sum_k A[i,k] * B[k,j]，这正是矩阵乘法。'
)

A = np.random.randn(3, 4)
B = np.random.randn(4, 5)
C1 = A @ B
C2 = np.einsum('ik,kj->ij', A, B)
print(f'A @ B 和 einsum 结果一致: {np.allclose(C1, C2)}')

print(f'矩阵转置: {np.allclose(np.einsum("ij->ji", A), A.T)}')
print(f'求和: {np.allclose(np.einsum("ij->", A), A.sum())}')
print(f'行求和: {np.allclose(np.einsum("ij->i", A), A.sum(axis=1))}')

### 第5题: np.linalg.norm 的用途

对于一个 m x n 的矩阵 A，np.linalg.norm(A, ord='fro') 计算的是什么？

A. 矩阵的最大奇异值 (谱范数)

B. 所有元素平方和的平方根 (Frobenius范数)

C. 所有行向量L2范数的最大值

D. 矩阵行列式的绝对值

In [ ]:
# 第5题
answer = 'A'  # 修改这里

check_answer(5, answer, 'B', 
    'Frobenius范数 = sqrt(sum(|a_ij|^2))，即所有元素平方和的平方根。'
    '常用范数: ord=None默认Frobenius, ord=2为谱范数, ord=1为列和最大值。'
)

A = np.array([[1, 2], [3, 4]], dtype=float)
fro_np = np.linalg.norm(A, 'fro')
fro_manual = np.sqrt(np.sum(A ** 2))
print(f'Frobenius范数 (numpy): {fro_np:.4f}')
print(f'Frobenius范数 (手动): {fro_manual:.4f}')
print(f'谱范数(最大奇异值): {np.linalg.norm(A, 2):.4f}')
print(f'L1范数(列和最大值): {np.linalg.norm(A, 1):.4f}')

### 第6题: np.linalg.svd 的含义

关于奇异值分解 A = U * Sigma * V^T，以下说法**错误**的是？

A. U 的列向量称为左奇异向量，V 的列向量称为右奇异向量

B. Sigma 是对角矩阵，对角线上的值称为奇异值，且非负递减排列

C. SVD 可以用于降维，取前 k 个奇异值做低秩近似

D. 对于方阵，奇异值就是特征值

In [ ]:
# 第6题
answer = 'A'  # 修改这里

check_answer(6, answer, 'D', 
    '奇异值不等于特征值! 奇异值是 A^T*A 特征值的平方根。'
    '只有当 A 是对称正定矩阵时，奇异值才等于特征值的绝对值。'
    'SVD适用于任意形状的矩阵(不限于方阵)，这是它比特征值分解更通用的原因。'
)

A = np.random.randn(4, 3)
U, S, Vt = np.linalg.svd(A, full_matrices=False)
print(f'A shape: {A.shape}')
print(f'U: {U.shape}, S: {S.shape}, Vt: {Vt.shape}')
print(f'奇异值: {S}')
print(f'重构误差: {np.linalg.norm(A - U @ np.diag(S) @ Vt):.2e}')

eigvals = np.linalg.eigvals(A.T @ A)
print(f'\nA^T*A 的特征值: {np.sort(eigvals)[::-1]}')
print(f'奇异值的平方: {S ** 2}')

### 第7题: 图像的傅里叶变换

关于图像的二维傅里叶变换，以下说法**错误**的是？

A. 低频分量对应图像中缓慢变化的部分 (如大面积色块)

B. 高频分量对应图像中快速变化的部分 (如边缘、噪声)

C. 傅里叶变换后，频率零点 (直流分量) 默认位于频谱图的中心

D. 低通滤波会模糊图像，高通滤波会增强边缘

In [ ]:
# 第7题
answer = 'A'  # 修改这里

check_answer(7, answer, 'C', 
    '傅里叶变换后，零频默认在频谱图的左上角，不是中心! '
    '需要使用 np.fft.fftshift 将零频移到中心。'
)

x = np.linspace(0, 1, 128)
signal = np.sin(2 * np.pi * 5 * x) + 0.5 * np.sin(2 * np.pi * 20 * x)
fft_result = np.fft.fft(signal)
freq = np.fft.fftfreq(len(signal), d=1/128)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, signal)
axes[0].set_title('原始信号')
axes[1].plot(freq[:64], np.abs(fft_result[:64]))
axes[1].set_title('频谱')
plt.tight_layout()
plt.show()

### 第8题: np.pad 的填充模式

对于数组 a = np.array([1, 2, 3, 4])，执行 np.pad(a, 2, mode='reflect') 的结果是？

A. [1, 1, 1, 2, 3, 4, 4, 4]

B. [3, 2, 1, 2, 3, 4, 3, 2]

C. [0, 0, 1, 2, 3, 4, 0, 0]

D. [2, 1, 1, 2, 3, 4, 4, 3]

In [ ]:
# 第8题
answer = 'A'  # 修改这里

check_answer(8, answer, 'B', 
    'reflect 模式是以边界为镜面进行反射: [1,2,3,4] 左边反射2个得到 3,2，'
    '右边反射2个得到 3,2。结果为 [3, 2, 1, 2, 3, 4, 3, 2]。'
)

a = np.array([1, 2, 3, 4])
print(f'reflect:  {np.pad(a, 2, mode="reflect")}')
print(f'constant: {np.pad(a, 2, mode="constant")}')
print(f'edge:     {np.pad(a, 2, mode="edge")}')
print(f'wrap:     {np.pad(a, 2, mode="wrap")}')

### 第9题: np.argmax / argmin

对于数组 a = np.array([[1, 5, 3], [4, 2, 6]])，np.argmax(a, axis=0) 的结果是？

A. [1, 0, 1]

B. [0, 1, 0]

C. [1, 0]

D. [3, 5]

In [ ]:
# 第9题
answer = 'A'  # 修改这里

check_answer(9, answer, 'A', 
    'axis=0 表示沿着第0轴(行方向)找最大值的索引，即每列的最大值在哪一行。'
    '第0列: max(1,4)=4在行1; 第1列: max(5,2)=5在行0; 第2列: max(3,6)=6在行1。'
)

a = np.array([[1, 5, 3], [4, 2, 6]])
print(f'argmax(axis=0): {np.argmax(a, axis=0)}')
print(f'argmax(axis=1): {np.argmax(a, axis=1)}')
print(f'argmax(): {np.argmax(a)}')

### 第10题: 卷积与互相关

在 CNN 中，我们通常说的卷积操作，数学上严格来说是？

A. 卷积 (convolution)，需要翻转卷积核

B. 互相关 (cross-correlation)，不翻转卷积核

C. 自相关 (autocorrelation)

D. 循环卷积 (circular convolution)

In [ ]:
# 第10题
answer = 'A'  # 修改这里

check_answer(10, answer, 'B', 
    'CNN中的卷积实际上是互相关 (cross-correlation)。'
    '数学上的卷积需要翻转核，但CNN中直接用核在图像上滑动做加权求和，不翻转。'
    '由于CNN的核参数是学习得来的，翻转与否等价于参数的重新排列，所以不影响结果。'
)

img = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=float)
kernel = np.array([[1, 0], [0, -1]], dtype=float)

# 互相关: 直接滑动 (CNN的做法)
cross_corr = np.array([
    [1*1 + 2*0 + 4*0 + 5*(-1), 2*1 + 3*0 + 5*0 + 6*(-1)],
    [4*1 + 5*0 + 7*0 + 8*(-1), 5*1 + 6*0 + 8*0 + 9*(-1)]
])

# 卷积: 先翻转180度再滑动
kernel_flipped = kernel[::-1, ::-1]
conv_result = np.array([
    [(-1)*1 + 0*2 + 0*4 + 1*5, (-1)*2 + 0*3 + 0*5 + 1*6],
    [(-1)*4 + 0*5 + 0*7 + 1*8, (-1)*5 + 0*6 + 0*8 + 1*9]
])

print(f'互相关结果:\n{cross_corr}')
print(f'卷积结果(核翻转后):\n{conv_result}')

### 第11题: 卷积的数学定义

一维连续卷积的定义是 (f*g)(t) = integral f(tau) * g(t-tau) d_tau，
对于离散序列 f = [1, 2, 3] 和 g = [0, 1, 0.5]，离散卷积 (f*g)[1] 的值是？

A. 2.0

B. 2.5

C. 1.5

D. 3.0

In [ ]:
# 第11题
answer = 'A'  # 修改这里

check_answer(11, answer, 'B', 
    '离散卷积: (f*g)[n] = sum_m f[m] * g[n-m]。'
    '直接用 numpy.convolve: np.convolve([1,2,3], [0,1,0.5]) = [0, 1, 2.5, 4, 1.5]。'
    '索引1的值为1，索引2的值为2.5。'
)

f = np.array([1, 2, 3])
g = np.array([0, 1, 0.5])
conv = np.convolve(f, g, mode='full')
print(f'完整卷积结果: {conv}')
print(f'索引0: {conv[0]}, 索引1: {conv[1]}, 索引2: {conv[2]}')

---

## 二、编程练习部分

---

### 编程题 1: 用 NumPy 实现图像的仿射变换

**要求**: 不使用 cv2.warpAffine，手动实现仿射变换。

仿射变换公式: dst(x', y') = src(x, y)，其中 [x', y', 1] = M @ [x, y, 1]^T

实际实现中需要**逆映射**: 对于输出图像中的每个像素 (x', y')，计算其在原图中的坐标，然后插值获取像素值。

- 支持平移、旋转、缩放
- 使用双线性插值

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def affine_transform(image, M, output_shape=None):
    """
    手动实现仿射变换
    
    参数:
        image: 输入图像 (H, W) 或 (H, W, C)
        M: 2x3 仿射变换矩阵
        output_shape: 输出图像大小 (out_H, out_W)
    
    返回:
        变换后的图像
    """
    # TODO: 实现仿射变换
    pass

# 测试
test_img = np.zeros((100, 100), dtype=np.float64)
test_img[30:70, 20:80] = 1.0

# TODO: 构造旋转45度的仿射矩阵并测试

In [ ]:
# ====== 参考答案 ======

def affine_transform(image, M, output_shape=None):
    """
    手动实现仿射变换 (逆映射 + 双线性插值)
    """
    h, w = image.shape[:2]
    if output_shape is None:
        output_shape = (h, w)
    out_h, out_w = output_shape
    
    # 逆变换矩阵
    M_full = np.vstack([M, [0, 0, 1]])
    M_inv = np.linalg.inv(M_full)
    M_inv_2x3 = M_inv[:2, :]
    
    # 生成输出像素坐标
    out_y, out_x = np.mgrid[0:out_h, 0:out_w]
    coords = np.stack([out_x.ravel(), out_y.ravel(), np.ones(out_h * out_w)])
    
    # 逆映射到原图坐标
    src_coords = M_inv_2x3 @ coords
    src_x = src_coords[0].reshape(out_h, out_w)
    src_y = src_coords[1].reshape(out_h, out_w)
    
    # 双线性插值
    x0 = np.floor(src_x).astype(int)
    y0 = np.floor(src_y).astype(int)
    x1 = x0 + 1
    y1 = y0 + 1
    wx = src_x - x0
    wy = src_y - y0
    
    valid = (x0 >= 0) & (x1 < w) & (y0 >= 0) & (y1 < h)
    x0c = np.clip(x0, 0, w - 1)
    y0c = np.clip(y0, 0, h - 1)
    x1c = np.clip(x1, 0, w - 1)
    y1c = np.clip(y1, 0, h - 1)
    
    if image.ndim == 2:
        result = (
            (1 - wx) * (1 - wy) * image[y0c, x0c] +
            wx * (1 - wy) * image[y0c, x1c] +
            (1 - wx) * wy * image[y1c, x0c] +
            wx * wy * image[y1c, x1c]
        )
        result[~valid] = 0
    else:
        c = image.shape[2]
        result = np.zeros((out_h, out_w, c), dtype=image.dtype)
        for ch in range(c):
            result[:, :, ch] = (
                (1 - wx) * (1 - wy) * image[y0c, x0c, ch] +
                wx * (1 - wy) * image[y0c, x1c, ch] +
                (1 - wx) * wy * image[y1c, x0c, ch] +
                wx * wy * image[y1c, x1c, ch]
            )
            result[:, :, ch][~valid] = 0
    
    return result


# 测试
test_img = np.zeros((100, 100), dtype=np.float64)
test_img[30:70, 20:80] = 1.0

angle = np.radians(45)
cx, cy = 50, 50
cos_a, sin_a = np.cos(angle), np.sin(angle)

M = np.array([
    [cos_a, -sin_a, cx * (1 - cos_a) + cy * sin_a],
    [sin_a,  cos_a, cy * (1 - cos_a) - cx * sin_a]
])

result = affine_transform(test_img, M)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(test_img, cmap='gray')
axes[0].set_title('原始图像')
axes[1].imshow(result, cmap='gray')
axes[1].set_title('旋转45度')
plt.tight_layout()
plt.show()

### 编程题 2: 手动实现 SVD 并用低秩近似压缩图像

**要求**:
1. 使用幂迭代法 (Power Iteration) 手动实现 SVD
2. 用低秩近似 (取前 k 个奇异值) 压缩灰度图像
3. 分析压缩比和近似误差的关系

In [ ]:
def manual_svd(A, k=None, n_iter=100):
    """
    使用幂迭代法手动计算 SVD
    参数: A (m,n), k=奇异值个数, n_iter=迭代次数
    返回: U, S, Vt
    """
    # TODO: 实现幂迭代 SVD
    pass

def low_rank_approximation(image, k):
    """用 SVD 低秩近似压缩图像"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def manual_svd(A, k=None, n_iter=100):
    """
    使用幂迭代法 + deflate 手动计算 SVD
    
    思路: A^T @ A 的主特征向量就是右奇异向量 v，
    然后 u = Av / ||Av||, sigma = ||Av||
    """
    m, n = A.shape
    if k is None:
        k = min(m, n)
    k = min(k, min(m, n))
    
    U = np.zeros((m, k))
    S = np.zeros(k)
    Vt = np.zeros((k, n))
    A_residual = A.copy().astype(np.float64)
    
    for i in range(k):
        v = np.random.randn(n)
        v /= np.linalg.norm(v)
        
        for _ in range(n_iter):
            v_new = A_residual.T @ (A_residual @ v)
            v = v_new / np.linalg.norm(v_new)
        
        u = A_residual @ v
        sigma = np.linalg.norm(u)
        if sigma > 1e-10:
            u /= sigma
        
        U[:, i] = u
        S[i] = sigma
        Vt[i, :] = v
        A_residual -= sigma * np.outer(u, v)
    
    return U, S, Vt


def low_rank_approximation(image, k):
    """用 numpy SVD 做低秩近似"""
    U, S, Vt = np.linalg.svd(image, full_matrices=False)
    approx = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    return approx, S


# 测试
np.random.seed(42)
h, w = 128, 128
x = np.linspace(-2, 2, w)
y = np.linspace(-2, 2, h)
X, Y = np.meshgrid(x, y)
synth_img = np.exp(-(X**2 + Y**2) / 2) + 0.3 * np.sin(3*X) * np.cos(3*Y)
synth_img = (synth_img - synth_img.min()) / (synth_img.max() - synth_img.min())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(synth_img, cmap='gray')
axes[0].set_title('原始图像')

for idx, k in enumerate([5, 20, 50]):
    approx, S = low_rank_approximation(synth_img, k)
    error = np.linalg.norm(synth_img - approx, 'fro') / np.linalg.norm(synth_img, 'fro')
    ratio = (h * w) / (k * (h + w + 1))
    axes[idx + 1].imshow(approx, cmap='gray')
    axes[idx + 1].set_title(f'k={k}, err={error:.4f}, ratio={ratio:.1f}x')

plt.tight_layout()
plt.show()

# 奇异值衰减曲线
_, S_all, _ = np.linalg.svd(synth_img, full_matrices=False)
plt.figure(figsize=(8, 4))
plt.semilogy(S_all[:60], 'b.-')
plt.xlabel('奇异值索引')
plt.ylabel('奇异值 (log scale)')
plt.title('奇异值衰减曲线')
plt.grid(True, alpha=0.3)
plt.show()

### 编程题 3: 实现 2D FFT 和频域滤波

**要求**:
1. 使用 np.fft.fft2 对图像做二维傅里叶变换
2. 实现理想低通和高通滤波器
3. 实现高斯低通和高通滤波器
4. 展示频域滤波的效果

In [ ]:
def ideal_lowpass_filter(shape, cutoff):
    """理想低通滤波器"""
    # TODO
    pass

def gaussian_lowpass_filter(shape, sigma):
    """高斯低通滤波器"""
    # TODO
    pass

def apply_freq_filter(image, filter_mask):
    """在频域应用滤波器"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def ideal_lowpass_filter(shape, cutoff):
    """理想低通滤波器"""
    h, w = shape
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    dist = np.sqrt((x - cx)**2 + (y - cy)**2)
    return (dist <= cutoff).astype(np.float64)

def gaussian_lowpass_filter(shape, sigma):
    """高斯低通滤波器"""
    h, w = shape
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    dist_sq = (x - cx)**2 + (y - cy)**2
    return np.exp(-dist_sq / (2 * sigma**2))

def apply_freq_filter(image, filter_mask):
    """在频域应用滤波器: FFT -> fftshift -> 滤波 -> ifftshift -> IFFT"""
    fft = np.fft.fft2(image)
    fft_shifted = np.fft.fftshift(fft)
    filtered = fft_shifted * filter_mask
    filtered_unshifted = np.fft.ifftshift(filtered)
    return np.real(np.fft.ifft2(filtered_unshifted))

# 测试
np.random.seed(42)
h, w = 128, 128
x = np.linspace(0, 4*np.pi, w)
y = np.linspace(0, 4*np.pi, h)
X, Y = np.meshgrid(x, y)
clean_img = (np.sin(X) * np.cos(Y) + 1) / 2
noisy_img = clean_img + 0.2 * np.random.randn(h, w)
noisy_img = np.clip(noisy_img, 0, 1)

lp = gaussian_lowpass_filter((h, w), sigma=15)
hp = 1.0 - gaussian_lowpass_filter((h, w), sigma=10)
denoised = apply_freq_filter(noisy_img, lp)
edges = apply_freq_filter(clean_img, hp)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes[0, 0].imshow(noisy_img, cmap='gray'); axes[0, 0].set_title('加噪图像')
axes[0, 1].imshow(lp, cmap='gray'); axes[0, 1].set_title('高斯低通')
axes[0, 2].imshow(denoised, cmap='gray'); axes[0, 2].set_title('去噪结果')
axes[1, 0].imshow(clean_img, cmap='gray'); axes[1, 0].set_title('原始图像')
axes[1, 1].imshow(hp, cmap='gray'); axes[1, 1].set_title('高斯高通')
axes[1, 2].imshow(edges, cmap='gray'); axes[1, 2].set_title('边缘检测')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

### 编程题 4: 用 np.einsum 实现批量矩阵乘法和注意力机制

**要求**:
1. 用 einsum 实现批量矩阵乘法
2. 用 einsum 实现自注意力机制 (Self-Attention) 的核心计算
3. 对比不同实现方式的性能

In [ ]:
def batch_matmul_einsum(A, B):
    """批量矩阵乘法: A @ B, A:(batch,m,k), B:(batch,k,n)"""
    # TODO
    pass

def self_attention(Q, K, V):
    """
    自注意力机制
    Q,K: (batch, seq_len, d_k), V: (batch, seq_len, d_v)
    返回: output, attention_weights
    """
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def batch_matmul_einsum(A, B):
    """批量矩阵乘法"""
    return np.einsum('bmk,bkn->bmn', A, B)

def self_attention(Q, K, V):
    """自注意力机制: softmax(QK^T / sqrt(d_k)) V"""
    d_k = Q.shape[-1]
    scores = np.einsum('bsk,btk->bst', Q, K) / np.sqrt(d_k)
    scores_max = scores.max(axis=-1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    attn_weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
    output = np.einsum('bss,bsd->bsd', attn_weights, V)
    return output, attn_weights

# 测试
np.random.seed(42)
batch, seq_len, d_k, d_v = 2, 4, 8, 16

A = np.random.randn(batch, 3, d_k)
B = np.random.randn(batch, d_k, 5)
result_einsum = batch_matmul_einsum(A, B)
result_np = A @ B
print(f'批量矩阵乘法验证: {np.allclose(result_einsum, result_np)}')

Q = np.random.randn(batch, seq_len, d_k)
K = np.random.randn(batch, seq_len, d_k)
V = np.random.randn(batch, seq_len, d_v)

output, attn_weights = self_attention(Q, K, V)
print(f'输出形状: {output.shape}')
print(f'注意力权重形状: {attn_weights.shape}')
print(f'注意力权重行和: {attn_weights.sum(axis=-1)}')

### 编程题 5: 实现图像直方图比较

**要求**: 实现三种直方图比较方法：
1. 相关性 (Correlation): 越接近1越相似
2. 卡方距离 (Chi-Square): 越小越相似
3. 巴氏距离 (Bhattacharyya Distance): 越小越相似，范围[0,1]

In [ ]:
def histogram_correlation(h1, h2):
    """直方图相关性"""
    # TODO
    pass

def histogram_chisquare(h1, h2):
    """卡方距离"""
    # TODO
    pass

def histogram_bhattacharyya(h1, h2):
    """巴氏距离"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def histogram_correlation(h1, h2):
    """直方图相关性, [-1,1], 越接近1越相似"""
    h1, h2 = h1.astype(float), h2.astype(float)
    d1, d2 = h1 - h1.mean(), h2 - h2.mean()
    denom = np.sqrt(np.sum(d1**2) * np.sum(d2**2))
    return np.sum(d1 * d2) / denom if denom > 1e-10 else 0.0

def histogram_chisquare(h1, h2):
    """卡方距离, >=0, 越小越相似"""
    h1, h2 = h1.astype(float), h2.astype(float)
    denom = h1 + h2
    mask = denom > 0
    return np.sum((h1[mask] - h2[mask])**2 / denom[mask])

def histogram_bhattacharyya(h1, h2):
    """巴氏距离, [0,1], 越小越相似"""
    h1, h2 = h1.astype(float), h2.astype(float)
    h1n = h1 / (h1.sum() + 1e-10)
    h2n = h2 / (h2.sum() + 1e-10)
    bc = min(np.sum(np.sqrt(h1n * h2n)), 1.0)
    return np.sqrt(1 - bc)

# 测试
n_bins = 256
x = np.arange(n_bins)
h1 = np.exp(-0.5 * ((x - 128) / 40)**2)
h1 = (h1 / h1.sum() * 10000).astype(int)
h2 = np.exp(-0.5 * ((x - 135) / 40)**2)
h2 = (h2 / h2.sum() * 10000).astype(int)
h3 = np.exp(-0.5 * ((x - 50) / 20)**2) + 0.5 * np.exp(-0.5 * ((x - 200) / 20)**2)
h3 = (h3 / h3.sum() * 10000).astype(int)

print(f'{"比较":<12} {"相关性":>8} {"卡方":>8} {"巴氏":>8}')
print('-' * 40)
for name, ha, hb in [('h1 vs h1', h1, h1), ('h1 vs h2', h1, h2), ('h1 vs h3', h1, h3)]:
    print(f'{name:<12} {histogram_correlation(ha,hb):>8.4f} '
          f'{histogram_chisquare(ha,hb):>8.2f} {histogram_bhattacharyya(ha,hb):>8.4f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(x, h1, width=1); axes[0].set_title('h1: 正态(mu=128)')
axes[1].bar(x, h2, width=1); axes[1].set_title('h2: 正态(mu=135)')
axes[2].bar(x, h3, width=1); axes[2].set_title('h3: 双峰')
plt.tight_layout()
plt.show()

---

## 三、面试要点总结

| 概念 | 关键点 |
|------|--------|
| shape/stride | stride是跨步字节数，转置只改stride不改内存 |
| Broadcasting | 从后往前比较，相同或一方为1 |
| dot/matmul/@/* | 前三个是矩阵乘法，* 是逐元素乘法 |
| einsum | 爱因斯坦求和约定，万能张量操作 |
| SVD | A=U*Sigma*V^T，奇异值是非负递减的，不等于特征值 |
| FFT | 零频默认在左上角，fftshift移到中心 |
| np.pad | constant/edge/reflect/wrap 四种常见模式 |
| CNN卷积 | 实际是互相关，不翻转核 |
| 范数 | L1/L2/Frobenius/谱范数各有用途 |